# Component ablation (VimQA n=500, Qwen2.5-7B 4-bit)

Answers Reviewer 3.1: what each component of the loop contributes. Every run turns off one component and is compared, question by question, against the default run `evidence_base_500.json` (same code, same 500 questions).

| Run | What is removed | Flag | Est. time (T4) |
|---|---|---|---|
| A1 | lexical half of the dual scorer (λ=1, dense only) | `--scorer dense` | ~45 min |
| A2 | semantic half of the dual scorer (λ=0, lexical only) | `--scorer lexical` | ~45 min |
| A3 | query reformulation: every pass reuses the original question | `--followup-mode none` | ~45 min |

The iteration budget (1 vs 5 passes) is already in `ablation_vimqa500_7b.json`.
Needs **Accelerator = GPU T4 x2**, **Internet = On**. Finished runs are skipped on re-run; set `JOB` to split the runs across accounts.

In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn bằng cách clone repo. Trước đây repo còn private nên notebook
# phải tải qua một gist công khai; repo public rồi, nên clone thẳng là nguồn duy
# nhất, không còn hai bản phải giữ đồng bộ.
import os, sys, subprocess, shutil

BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
GIT_URL = 'https://github.com/nhantrnh/ANLP-IterCOMP.git'
# 'budget' nằm trong danh sách này từ một bản cũ; module đó không tồn tại ở
# repo nào, nên cửa chặn luôn báo thiếu và kernel chết trước khi chạy gì.
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data', 'metrics', 'fertility', 'stats', 'stopping']

os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]

subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)

miss = [m for m in NEED
        if not os.path.exists(os.path.join(REPO, 'src', 'itercomp', m + '.py'))]
if miss:
    raise SystemExit('Bản clone thiếu module: ' + ', '.join(miss))

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

sha = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print('đủ', len(NEED), 'module | commit:', sha)


In [ ]:
import urllib.request
p = 'data/vimqa/validation.parquet'
os.makedirs(os.path.dirname(p), exist_ok=True)
if not os.path.exists(p):
    urllib.request.urlretrieve('https://huggingface.co/datasets/nguyenlab/vimqa/resolve/main/'
                               'data/validation-00000-of-00001.parquet', p)
# The new flag and the paired baseline must be in the clone; if not, src/ was not pushed.
assert '"none"' in open('src/scripts/run_eval.py', encoding='utf-8').read(), 'run_eval.py has no --followup-mode none: push src/ first.'
assert os.path.exists('src/results/evidence_base_500.json'), 'baseline evidence_base_500.json missing from the clone'
print('✓ data, code and baseline ready')

In [ ]:
import subprocess, sys, time
# JOB picks the runs, to split them across accounts: 'all' | 'A1' | 'A2' | 'A3'
JOB = 'all'
R7 = 'Qwen/Qwen2.5-7B-Instruct'
N = '500'

def run(cmd, log):
    t0 = time.time()
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

# Same arguments as the baseline run E0, so the only difference is the ablated component.
BASE_ARGS = [sys.executable, '-u', 'src/scripts/run_eval.py', '--dataset', 'vimqa', '--limit', N,
             '--reader', 'hf', '--reader-model', R7, '--load-4bit', '--itercomp-llm', 'hf',
             '--lam', '0.6', '--max-iter', '5', '--methods', 'itercomp']
RUNS = {
    'A1': ('src/results/ablation_dense_500.json',      ['--scorer', 'dense']),
    'A2': ('src/results/ablation_lexical_500.json',    ['--scorer', 'lexical']),
    'A3': ('src/results/ablation_nofollowup_500.json', ['--scorer', 'dual', '--followup-mode', 'none']),
}
for name, (out, extra) in RUNS.items():
    if JOB not in ('all', name): continue
    if os.path.exists(out): print('[skip]', out); continue
    print('=' * 60, '\n', name, extra, flush=True)
    assert run(BASE_ARGS + extra + ['--out', out], f'{name}.log') == 0, f'{name} failed'

In [ ]:
import shutil
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
for out, _ in RUNS.values():
    if os.path.exists(out): shutil.copy(out, BASE); print('✓', os.path.basename(out))

In [ ]:
# Paired comparison against the default run E0: F1*, F1, retention, passes, gold-evidence recall
import json, numpy as np
sys.path.insert(0, 'src')
from itercomp import load_dataset
rows = load_dataset('vimqa', int(N))

def gold_set(r):
    sf = r['supporting_facts']
    have = {(t, i) for t, ss in zip(r['context']['title'], r['context']['sentences'])
            for i, x in enumerate(ss) if x.strip()}
    return set(zip(sf['title'], sf['sent_id'])) & have

def boot(d, B=10000, seed=0):
    d = np.asarray(d, float); rng = np.random.default_rng(seed)
    m = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(B)]
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

def per_q(path):
    pr = json.load(open(path, encoding='utf-8'))['per_row']
    assert [p['question'] for p in pr] == [r['question'] for r in rows[:len(pr)]]
    out = []
    for p, r in zip(pr, rows):
        m = p['methods']['itercomp']; g = gold_set(r); sel = {tuple(x) for x in m['selected']}
        out.append(dict(f1n=100 * m['f1_norm'], f1=100 * m['f1'], ratio=m['tokens'] / max(p['orig_tokens'], 1),
                        it=m['iterations'], rec=100 * len(g & sel) / len(g) if g else np.nan))
    return out

base = per_q('src/results/evidence_base_500.json')
summary = {}
print(f"{'run':<6}{'F1*':>7}{'ΔF1* [95% CI]':>26}{'F1':>7}{'ΔF1':>8}{'ratio':>7}{'passes':>7}{'recall':>8}{'Δrecall':>9}")
print(f"{'E0':<6}{np.mean([b['f1n'] for b in base]):7.2f}{'':>26}{np.mean([b['f1'] for b in base]):7.2f}{'':>8}"
      f"{np.mean([b['ratio'] for b in base]):7.3f}{np.mean([b['it'] for b in base]):7.2f}{np.nanmean([b['rec'] for b in base]):8.1f}")
for name, (out, _) in RUNS.items():
    if not os.path.exists(out): print(f'{name:<6} (missing)'); continue
    a = per_q(out)
    d, lo, hi = boot([x['f1n'] - b['f1n'] for x, b in zip(a, base)])
    df1 = np.mean([x['f1'] - b['f1'] for x, b in zip(a, base)])
    dr = [x['rec'] - b['rec'] for x, b in zip(a, base) if not np.isnan(x['rec'])]
    r_m, r_lo, r_hi = boot(dr)
    summary[name] = dict(f1n=np.mean([x['f1n'] for x in a]), d_f1n=d, ci=[lo, hi], d_f1=df1,
                         ratio=np.mean([x['ratio'] for x in a]), passes=np.mean([x['it'] for x in a]),
                         recall=np.nanmean([x['rec'] for x in a]), d_recall=r_m, recall_ci=[r_lo, r_hi])
    s = summary[name]
    print(f"{name:<6}{s['f1n']:7.2f}{f'{d:+.2f} [{lo:+.2f},{hi:+.2f}]':>26}{np.mean([x['f1'] for x in a]):7.2f}{df1:+8.2f}"
          f"{s['ratio']:7.3f}{s['passes']:7.2f}{s['recall']:8.1f}{r_m:+9.1f}")
json.dump(summary, open(os.path.join(BASE, 'ablation_components_summary_500.json'), 'w'), indent=1)